# IRS Form 990 - Bulk Download & ProPublica API

In [98]:
import xml.etree.ElementTree as ET
from pathlib import Path
import json
import re
import requests
import pandas as pd
from utility import propublica_endpoint

In [111]:
# Base path
curr_dir = Path()

# Form990 folders
form990_dir = curr_dir / 'Form990'
form990_input_dir = form990_dir / 'input'
form990_input_csv_dir = form990_input_dir / 'csv'
form990_input_xml_dir = form990_input_dir / 'xml'
form990_output_dir = form990_dir / 'output'

In [106]:
filer = []
business_officer = []
return_data = []
employee_data = []
full_break = False

for xml_file in form990_input_xml_dir.glob('*.xml'):
    # Obtain root of xml file
    xml = ET.parse(xml_file)
    root = xml.getroot()

    # Acquire namespace, if there
    try:
        namespace = re.search(r'\{.+\}', root.tag).group()
    except AttributeError:
        namespace = ''
    
    # xml paths
    xml_path_ReturnHeader = f'./{namespace}ReturnHeader/'
    xml_path_PreparerFirmGrp = xml_path_ReturnHeader + f'{namespace}PreparerFirmGrp/'
    xml_path_Filer = xml_path_ReturnHeader + f'{namespace}Filer'
    xml_path_BusinessOfficerGrp = xml_path_ReturnHeader + f'{namespace}BusinessOfficerGrp'
    xml_path_ReturnData = f'./{namespace}ReturnData'

    filer_content = {}
    business_officer_content = {}
    return_content = {}

    # * Filer
    # Expecting shallow nesting
    for filer_item in root.find(xml_path_Filer):
        if len(filer_item) >= 1:
            # TODO: Handle instances where item text is nothing as it's only a parent
            # TODO: Figure out why 'BusinessName' is added but not 'USAddress'
            for filer_subitem in filer_item:
                key = re.sub(namespace, '', filer_subitem.tag)
                filer_content[key] = filer_subitem.text

            continue

        key = re.sub(namespace, '', filer_item.tag)
        filer_content[key] = filer_item.text

    filer.append(filer_content)

    # TODO: Implement preparer gathering
    # # Get preparer EIN if present; we only expect a max of one
    # field = 'PreparerFirmEIN'
    # try:
    #     filer_content[field] = root.find(f'{xml_path_PreparerFirmGrp}{namespace}{field}').text
    # except AttributeError:
    #     filer_content[field] = None

    # * BusinessOfficerGrp
    for business_officer_grp_item in root.find(xml_path_BusinessOfficerGrp):
        business_officer_content['EIN'] = filer_content['EIN'] # Include FK reference
        if len(business_officer_grp_item) >= 1:
            for business_officer_grp_subitem in filer_item:
                key = re.sub(namespace, '', business_officer_grp_subitem.tag)
                business_officer_content[key] = business_officer_grp_subitem.text
            continue

        key = re.sub(namespace, '', business_officer_grp_item.tag)
        business_officer_content[key] = business_officer_grp_item.text

    business_officer.append(business_officer_content)

    # * ReturnData
    # Assign FK column
    return_content['EIN'] = filer_content['EIN']
    # Also pulling TaxYr and TaxPeriodBeginDt from ReturnHeader section as it seems relevant
    for field in ['TaxYr', 'TaxPeriodEndDt']:
        return_content[field] = root.find(f'{xml_path_ReturnHeader}{namespace}{field}').text

    # TODO: parse documentId class/id in element (e.g., "RetDoc1038000001")
    for return_data_section in root.find(xml_path_ReturnData):
        # e.g., IRS990, IRS990ScheduleA, ...
        section = re.sub(namespace, '', return_data_section.tag)
        
        if section == 'IRS990ScheduleB':
            # 501(c)(4)s don't disclose contributors publicly after 2019
            # https://www.shipmangoodwin.com/insights/irs-finalizes-form-990-donor-disclosure-tax-rules.html
            # TODO: Handle this information for older files that would have this information:
                # e.g., if TaxYr < 2019...
            continue

        return_content[section] = {}
        
        for return_data_field in return_data_section:
            data_field_key = re.sub(namespace, '', return_data_field.tag)
            if data_field_key == 'Form990PartVIISectionAGrp':
                employee_content = {}
                for employee_field in return_data_field:
                    # Attach FK reference
                    employee_content['EIN'] = filer_content['EIN']
                    employee_content['TaxYr'] = return_content['TaxYr']

                    employee_field_key = re.sub(namespace, '', employee_field.tag)
                    employee_content[employee_field_key] = employee_field.text
                    
                employee_data.append(employee_content)
                continue

            # E.g., 'DoingBusinessAsName', 'USAddress', ...
            if len(return_data_field) >= 1:
                for return_data_subfield in return_data_field:
                    # This should handle instances where child element is just 'TotalAmt', or any other generic tag
                    data_subfield_key = data_field_key + '_' + re.sub(namespace, '', return_data_subfield.tag)
                    return_content[section][data_subfield_key] = return_data_subfield.text
                continue

            return_content[section][data_field_key] = return_data_field.text


    return_data.append(return_content)

In [109]:
# Convert to dataframes where possible
filer = pd.DataFrame(filer)
business_officer = pd.DataFrame(business_officer)
return_data = pd.DataFrame(return_data)
employee_data = pd.DataFrame(employee_data)

In [124]:
# * Filer export
filer.to_csv(
    form990_input_csv_dir / f'form990_filer_sample.csv'
)

# * BusinessOfficer export
business_officer.to_csv(
    form990_input_csv_dir / f'form990_business_officer_sample.csv'
)

# * ReturnData export
schedules = return_data.columns[3:]

for schedule in schedules:
    schedule_data = return_data.loc[return_data[schedule].notna(), :]
    schedule_df = pd.concat(
        [
            schedule_data.iloc[:, [0, 1, 2]], # Id columns
            pd.json_normalize(schedule_data.loc[:, schedule])
        ],
        axis=1
    )

    # Don't need redundant identifiers
    if schedule != 'IRS990':
        schedule = re.sub('IRS990', '', schedule)

    # More generalize file organization
    if schedule[:-1] == 'Schedule':
        schedule_folder = form990_input_csv_dir / 'CommonSchedules'
    elif schedule in ['EZ', 'PF', 'T'] or schedule == 'TScheduleA':
        schedule_folder = form990_input_csv_dir / 'AlternativeFormTypes'
    elif 'Sch' in schedule: # Should handle full Schedule spelling and 'SubstantialContributorsSch' example
        schedule_folder = form990_input_csv_dir / 'OtherSchedules'
    elif schedule not in ['IRS990', 'IRS4562']:
        schedule_folder = form990_input_csv_dir / 'Other'
    
    # TODO: Clean up the conditionals
    if schedule not in ['IRS990', 'IRS4562']:
        # Handle cases where directory has not yet been made
        try:
            schedule_df.to_csv(
                schedule_folder / f'form990_{schedule}_sample.csv'
            )
        except OSError: # No directory
            Path.mkdir(schedule_folder)
            schedule_df.to_csv(
                schedule_folder / f'form990_{schedule}_sample.csv'
            )
    else:
        schedule_df.to_csv(
                form990_input_csv_dir / f'form990_{schedule}_sample.csv'
            )

# * BusinessOfficer export
employee_data.to_csv(
    form990_input_csv_dir / f'form990_employee_data_sample.csv'
)

In [50]:
# sample_eins = pd.DataFrame(
#     filer_and_preparer_eins,
#     columns = [
#         'filer_ein',
#         'preparer_ein'
#     ]
# )

# sample_eins.head()

,filer_ein,preparer_ein
0,731102960,731589604
1,934867692,454570537
2,203718726,822396457
3,822204282,NaN
4,830317641,208019714


In [51]:
# sample_eins.to_csv(form990_input_dir / 'sample_eins.csv')

In [4]:
sample_eins = pd.read_csv(
    form990_input_dir / 'sample_eins.csv'
).iloc[:, [1, 2]]

sample_eins.head()

,filer_ein,preparer_ein
0,731102960,731589604.0
1,934867692,454570537.0
2,203718726,822396457.0
3,822204282,NaN
4,830317641,208019714.0


In [8]:
for filer_ein in sample_eins.loc[:, 'filer_ein']:
    url = propublica_endpoint('organization', str(filer_ein))
    organization = requests.get(
        url
    ).json()

    print(json.dumps(organization, indent=4))

    break

{
    "organization": {
        "id": 731102960,
        "ein": 731102960,
        "name": "Bright Sky Inc",
        "careofname": "% SHANNA RICE",
        "address": "PO BOX 829",
        "city": "Chickasha",
        "state": "OK",
        "zipcode": "73023-0829",
        "exemption_number": 0,
        "subsection_code": 3,
        "affiliation_code": 3,
        "classification_codes": "1000",
        "ruling_date": "1983-09-01",
        "deductibility_code": 1,
        "foundation_code": 17,
        "activity_codes": "602328000",
        "organization_code": 1,
        "exempt_organization_status_code": 1,
        "tax_period": "2025-06-01",
        "asset_code": 5,
        "income_code": 2,
        "filing_requirement_code": 2,
        "pf_filing_requirement_code": 0,
        "accounting_period": 6,
        "asset_amount": 977248,
        "income_amount": 24861,
        "revenue_amount": 24861,
        "ntee_code": "O20Z",
        "sort_name": null,
        "created_at": "2023-05-09